<a href="https://colab.research.google.com/github/sbattineni3276/clickbaitdetectionBERT/blob/main/CS376_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
# BERT Clickbait Detection Project
## Setup and Installation

First, let's install the required packages and check GPU availability.
"""

!pip install transformers datasets accelerate scikit-learn -q

import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
print("=" * 50)
print("SYSTEM INFORMATION")
print("=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 50)

SYSTEM INFORMATION
PyTorch Version: 2.9.0+cpu
GPU Available: False


In [ ]:
"""
## Mount Google Drive

This allows us to access data and save results persistently.
Place your dataset in: `/content/drive/MyDrive/clickbait_data/`
"""

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
## Project Configuration

Set all hyperparameters and paths here for easy experimentation.
"""
class Config:
    # Paths (UPDATE THESE TO YOUR PATHS)
    INSTANCES_PATH = '/content/drive/MyDrive/clickbait_data/instances.jsonl'
    TRUTH_PATH = '/content/drive/MyDrive/clickbait_data/truth.jsonl'
    OUTPUT_DIR = '/content/drive/MyDrive/clickbait_results'
    MODEL_SAVE_PATH = '/content/drive/MyDrive/clickbait_model'

    # Model settings
    MODEL_NAME = 'bert-base-uncased'
    MAX_LENGTH = 512

    # Training settings
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 3
    WEIGHT_DECAY = 0.01

    # Data settings
    TEST_SIZE = 0.2
    RANDOM_SEED = 42
    CLICKBAIT_THRESHOLD = 0.5

    # Input configuration: 'title', 'body', or 'combined'
    INPUT_TYPE = 'title'  # Change this to test different inputs

config = Config()

print("Configuration:")
print(f"  Model: {config.MODEL_NAME}")
print(f"  Input Type: {config.INPUT_TYPE}")
print(f"  Batch Size: {config.BATCH_SIZE}")
print(f"  Epochs: {config.NUM_EPOCHS}")
print(f"  Max Length: {config.MAX_LENGTH}")

Configuration:
  Model: bert-base-uncased
  Input Type: title
  Batch Size: 16
  Epochs: 3
  Max Length: 512


In [ ]:
# ====================
# CELL 4: Data Loading
# ====================
"""
## Load and Explore Data

Loading the Webis Clickbait Corpus 2017 dataset.
The dataset comes in two files: instances.jsonl and truth.jsonl
"""

import json

print("Loading instances and truth files...")

# Load instances (contains text data)
instances = []
with open('/content/drive/MyDrive/clickbait_data/instances.jsonl', 'r') as f:
    for line in f:
        instances.append(json.loads(line))
instances_df = pd.DataFrame(instances)

print(f"Instances loaded: {len(instances_df)} samples")
print(f"Instances columns: {instances_df.columns.tolist()}")

# Load truth (contains clickbait scores/labels)
truth = []
with open('/content/drive/MyDrive/clickbait_data/truth.jsonl', 'r') as f:
    for line in f:
        truth.append(json.loads(line))
truth_df = pd.DataFrame(truth)

print(f"\nTruth loaded: {len(truth_df)} samples")
print(f"Truth columns: {truth_df.columns.tolist()}")

# Merge on ID
df = instances_df.merge(truth_df, on='id', how='inner')

print(f"\nMerged dataset: {len(df)} samples")
print(f"\nAll columns: {df.columns.tolist()}")
print(f"\nFirst row preview:")
print(df.head(1).to_dict('records')[0])

# Check for required data
if 'postText' in df.columns:
    print(f"\n✓ postText found")
if 'truthMean' in df.columns:
    print(f"✓ truthMean found")
elif 'truthClass' in df.columns:
    print(f"✓ truthClass found (will convert to truthMean)")
    # If only truthClass exists, convert it
    df['truthMean'] = df['truthClass'].map({'clickbait': 1.0, 'no-clickbait': 0.0})

Loading instances and truth files...
Instances loaded: 19538 samples
Instances columns: ['postMedia', 'postText', 'id', 'targetCaptions', 'targetParagraphs', 'targetTitle', 'postTimestamp', 'targetKeywords', 'targetDescription']

Truth loaded: 19538 samples
Truth columns: ['truthJudgments', 'truthMean', 'id', 'truthClass', 'truthMedian', 'truthMode']

Merged dataset: 19538 samples

All columns: ['postMedia', 'postText', 'id', 'targetCaptions', 'targetParagraphs', 'targetTitle', 'postTimestamp', 'targetKeywords', 'targetDescription', 'truthJudgments', 'truthMean', 'truthClass', 'truthMedian', 'truthMode']

First row preview:
{'postMedia': [], 'postText': ['UK’s response to modern slavery leaving victims destitute while abusers go free'], 'id': '858462320779026433', 'targetCaptions': ['modern-slavery-rex.jpg'], 'targetParagraphs': ['Thousands of modern slavery victims have\xa0not come forward, while others who have chosen to report their enslavers have ended up destitute as a result of i

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ====================
# CELL 5: Data Preprocessing
# ====================
"""
## Data Preprocessing

1. Create binary labels from truthMean scores
2. Prepare text inputs based on INPUT_TYPE configuration
3. Handle missing values
"""

# Helper function to safely extract text from various formats
def safe_text_extract(x):
    """Extract text from string, list, or other formats"""
    if isinstance(x, str):
        return x
    elif isinstance(x, list):
        if len(x) > 0 and isinstance(x[0], str):
            return x[0]
        else:
            return ' '.join(str(item) for item in x if item)
    elif x is None or pd.isna(x):
        return ''
    else:
        return str(x)

def safe_paragraphs_extract(x, num_paragraphs=3):
    """Extract paragraphs safely"""
    if isinstance(x, list):
        return ' '.join(str(p) for p in x[:num_paragraphs] if p)
    elif isinstance(x, str):
        return x
    elif x is None or pd.isna(x):
        return ''
    else:
        return str(x)

# Extract postText from array/string format
print("Processing postText field...")
df['postText'] = df['postText'].apply(safe_text_extract)

# Process targetParagraphs if it exists
if 'targetParagraphs' in df.columns:
    print("Processing targetParagraphs field...")
    df['targetParagraphs_text'] = df['targetParagraphs'].apply(
        lambda x: safe_paragraphs_extract(x, num_paragraphs=3)
    )

# Create binary labels
df['label'] = (df['truthMean'] >= config.CLICKBAIT_THRESHOLD).astype(int)

# Create text inputs based on configuration
if config.INPUT_TYPE == 'title':
    df['text'] = df['postText']
    print("Using: Title only")

elif config.INPUT_TYPE == 'body':
    # Use first 3 paragraphs of article body
    if 'targetParagraphs' in df.columns:
        df['text'] = df['targetParagraphs_text']
    else:
        print("WARNING: targetParagraphs not found, using postText instead")
        df['text'] = df['postText']
    print("Using: Article body only")

elif config.INPUT_TYPE == 'combined':
    # Combine title and body with [SEP] token
    if 'targetParagraphs' in df.columns:
        df['text'] = df['postText'] + ' [SEP] ' + df['targetParagraphs_text']
    else:
        print("WARNING: targetParagraphs not found, using postText only")
        df['text'] = df['postText']
    print("Using: Title + Body combined")

# Remove any rows with missing text
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.len() > 0]

print(f"\nAfter preprocessing: {len(df)} samples")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nClickbait ratio: {df['label'].mean():.2%}")

# Show sample texts
print("\n" + "="*50)
print("SAMPLE NON-CLICKBAIT:")
print(df[df['label']==0]['text'].iloc[0][:200] + "...")
print("\n" + "="*50)
print("SAMPLE CLICKBAIT:")
print(df[df['label']==1]['text'].iloc[0][:200] + "...")
print("="*50)


Processing postText field...
Processing targetParagraphs field...
Using: Title only

After preprocessing: 19484 samples

Class distribution:
label
0    14788
1     4696
Name: count, dtype: int64

Clickbait ratio: 24.10%

SAMPLE NON-CLICKBAIT:
UK’s response to modern slavery leaving victims destitute while abusers go free...

SAMPLE CLICKBAIT:
this is good...


In [ ]:
# ====================
# CELL 6: Train/Test Split
# ====================
"""
## Create Train/Test Split

Stratified split to maintain class balance.
"""

train_df, test_df = train_test_split(
    df[['text', 'label']],
    test_size=config.TEST_SIZE,
    stratify=df['label'],
    random_state=config.RANDOM_SEED
)

print(f"Training set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"\nTrain class distribution:")
print(train_df['label'].value_counts())
print(f"\nTest class distribution:")
print(test_df['label'].value_counts())

Training set: 15587 samples
Test set: 3897 samples

Train class distribution:
label
0    11830
1     3757
Name: count, dtype: int64

Test class distribution:
label
0    2958
1     939
Name: count, dtype: int64


In [ ]:
# ====================
# CELL 7: Tokenization
# ====================
"""
## Tokenization

Convert text to BERT-compatible input format.
"""

from transformers import AutoTokenizer
from datasets import Dataset

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=config.MAX_LENGTH
    )

print("Creating datasets...")
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print("Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Setting format for PyTorch...")
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"\nTokenization complete!")
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Creating datasets...
Tokenizing...


Map:   0%|          | 0/15587 [00:00<?, ? examples/s]

Map:   0%|          | 0/3897 [00:00<?, ? examples/s]

Setting format for PyTorch...

Tokenization complete!
Training samples: 15587
Test samples: 3897


In [ ]:

# ====================
# CELL 8: Define Evaluation Metrics
# ====================
"""
## Evaluation Metrics Function

Computes all required metrics:
- Accuracy
- Macro F1
- ROC AUC
- Per-class Precision and Recall
"""

def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for the model.

    Returns:
        dict: Dictionary containing all evaluation metrics
    """
    predictions, labels = eval_pred

    # Convert logits to probabilities
    probs = torch.softmax(torch.tensor(predictions), dim=-1).numpy()

    # Get predicted classes
    preds = np.argmax(predictions, axis=1)

    # Calculate metrics
    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro')
    roc_auc = roc_auc_score(labels, probs[:, 1])

    # Per-class precision and recall
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'roc_auc': roc_auc,
        'precision_non_clickbait': precision[0],
        'recall_non_clickbait': recall[0],
        'f1_non_clickbait': f1[0],
        'precision_clickbait': precision[1],
        'recall_clickbait': recall[1],
        'f1_clickbait': f1[1],
    }

print("Metrics function defined!")


Metrics function defined!


In [ ]:
# ====================
# CELL 9: Load Model
# ====================
"""
## Load Pre-trained BERT Model

Loading BERT for sequence classification with 2 output classes.
"""

from transformers import AutoModelForSequenceClassification

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=2
)

print(f"Model loaded: {config.MODEL_NAME}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading model...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: bert-base-uncased
Number of parameters: 109,483,778


In [ ]:
# Update transformers to latest version
!pip install --upgrade transformers accelerate -q

# Restart may be needed - if you get errors after this, restart runtime
print("✓ Transformers updated! If you see errors, go to Runtime > Restart runtime")

✓ Transformers updated! If you see errors, go to Runtime > Restart runtime


In [ ]:
# ====================
# CELL 10: Training Configuration
# ====================
"""
## Configure Training

Set up the Trainer with all hyperparameters.
"""

from transformers import TrainingArguments, Trainer
import transformers

print(f"Transformers version: {transformers.__version__}")

# Use compatible parameter names for different versions
training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    eval_strategy='epoch',  # Changed from evaluation_strategy
    save_strategy='epoch',
    learning_rate=config.LEARNING_RATE,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    num_train_epochs=config.NUM_EPOCHS,
    weight_decay=config.WEIGHT_DECAY,
    logging_dir=f'{config.OUTPUT_DIR}/logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=2,
    report_to='none',  # Disable wandb/tensorboard
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer configured successfully!")
print(f"\nTraining will run for {config.NUM_EPOCHS} epochs")
print(f"Total training steps: {len(train_dataset) * config.NUM_EPOCHS // config.BATCH_SIZE}")

Transformers version: 4.57.3
Trainer configured successfully!

Training will run for 3 epochs
Total training steps: 2922


In [ ]:

# ====================
# CELL 11: Train Model
# ====================
"""
## Train the Model

This will take several minutes depending on your GPU.
Progress bars will show training progress.
"""

print("Starting training...")
print("=" * 50)

train_result = trainer.train()

print("\n" + "=" * 50)
print("TRAINING COMPLETE!")
print("=" * 50)
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training samples/second: {train_result.metrics['train_samples_per_second']:.2f}")

Starting training...


In [ ]:

# ====================
# CELL 12: Evaluate on Test Set
# ====================
"""
## Final Evaluation on Test Set

Comprehensive evaluation with all metrics.
"""

print("Evaluating on test set...")
print("=" * 50)

# Get predictions
predictions = trainer.predict(test_dataset)

# Compute metrics
test_metrics = compute_metrics((predictions.predictions, predictions.label_ids))

# Display results
print("\nTEST SET PERFORMANCE")
print("=" * 50)
print(f"Accuracy:      {test_metrics['accuracy']:.4f}")
print(f"Macro F1:      {test_metrics['macro_f1']:.4f}")
print(f"ROC AUC:       {test_metrics['roc_auc']:.4f}")
print("\nPer-Class Metrics:")
print("-" * 50)
print("Non-Clickbait (Class 0):")
print(f"  Precision:   {test_metrics['precision_non_clickbait']:.4f}")
print(f"  Recall:      {test_metrics['recall_non_clickbait']:.4f}")
print(f"  F1-Score:    {test_metrics['f1_non_clickbait']:.4f}")
print("\nClickbait (Class 1):")
print(f"  Precision:   {test_metrics['precision_clickbait']:.4f}")
print(f"  Recall:      {test_metrics['recall_clickbait']:.4f}")
print(f"  F1-Score:    {test_metrics['f1_clickbait']:.4f}")
print("=" * 50)

Evaluating on test set...


NameError: name 'trainer' is not defined

In [ ]:
# ====================
# CELL 13: Confusion Matrix
# ====================
"""
## Confusion Matrix

Visualize prediction patterns.
"""

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Get predictions
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Create confusion matrix
cm = confusion_matrix(labels, preds)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Clickbait', 'Clickbait'],
            yticklabels=['Non-Clickbait', 'Clickbait'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - Clickbait Detection')
plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved to: {config.OUTPUT_DIR}/confusion_matrix.png")


NameError: name 'np' is not defined

In [ ]:

# ====================
# CELL 14: Save Model and Results
# ====================
"""
## Save Model and Results

Save everything to Google Drive for future use.
"""

import os
import json

# Create output directory if it doesn't exist
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.MODEL_SAVE_PATH, exist_ok=True)

# Save model and tokenizer
print("Saving model and tokenizer...")
model.save_pretrained(config.MODEL_SAVE_PATH)
tokenizer.save_pretrained(config.MODEL_SAVE_PATH)
print(f"Model saved to: {config.MODEL_SAVE_PATH}")

# Save metrics to JSON
metrics_dict = {
    'config': {
        'model_name': config.MODEL_NAME,
        'input_type': config.INPUT_TYPE,
        'max_length': config.MAX_LENGTH,
        'batch_size': config.BATCH_SIZE,
        'learning_rate': config.LEARNING_RATE,
        'num_epochs': config.NUM_EPOCHS,
    },
    'test_metrics': {k: float(v) for k, v in test_metrics.items()},
    'training_time': train_result.metrics['train_runtime'],
}

with open(f'{config.OUTPUT_DIR}/results.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print(f"Results saved to: {config.OUTPUT_DIR}/results.json")

# Save detailed results to CSV
results_df = pd.DataFrame([{
    'Model': config.MODEL_NAME,
    'Input_Type': config.INPUT_TYPE,
    'Accuracy': test_metrics['accuracy'],
    'Macro_F1': test_metrics['macro_f1'],
    'ROC_AUC': test_metrics['roc_auc'],
    'Precision_Non_Clickbait': test_metrics['precision_non_clickbait'],
    'Recall_Non_Clickbait': test_metrics['recall_non_clickbait'],
    'Precision_Clickbait': test_metrics['precision_clickbait'],
    'Recall_Clickbait': test_metrics['recall_clickbait'],
}])

results_df.to_csv(f'{config.OUTPUT_DIR}/metrics_summary.csv', index=False)
print(f"Metrics summary saved to: {config.OUTPUT_DIR}/metrics_summary.csv")

print("\n✓ All results saved successfully!")

Saving model and tokenizer...
Model saved to: /content/drive/MyDrive/clickbait_model
Results saved to: /content/drive/MyDrive/clickbait_results/results.json
Metrics summary saved to: /content/drive/MyDrive/clickbait_results/metrics_summary.csv

✓ All results saved successfully!


In [ ]:

# ====================
# CELL 15: Example Predictions
# ====================
"""
## Test with Example Predictions

Try the model on new examples.
"""

def predict_clickbait(text):
    """
    Predict whether a text is clickbait.

    Args:
        text (str): Input text to classify

    Returns:
        dict: Prediction results
    """
    # Tokenize
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                      max_length=config.MAX_LENGTH, padding=True)

    # Move to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model.cuda()

    # Get prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1)

    return {
        'text': text,
        'prediction': 'Clickbait' if pred.item() == 1 else 'Non-Clickbait',
        'confidence': probs[0][pred].item(),
        'clickbait_probability': probs[0][1].item(),
        'non_clickbait_probability': probs[0][0].item(),
    }

# Test examples
examples = [
    "You Won't Believe What Happened Next!",
    "New Study Shows Benefits of Regular Exercise",
    "10 Shocking Celebrity Secrets They Don't Want You To Know!",
    "Federal Reserve Announces Interest Rate Decision"
]

print("EXAMPLE PREDICTIONS")
print("=" * 70)
for example in examples:
    result = predict_clickbait(example)
    print(f"\nText: {result['text']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"  - Clickbait probability: {result['clickbait_probability']:.2%}")
    print(f"  - Non-clickbait probability: {result['non_clickbait_probability']:.2%}")
    print("-" * 70)

EXAMPLE PREDICTIONS

Text: You Won't Believe What Happened Next!
Prediction: Clickbait
Confidence: 96.24%
  - Clickbait probability: 96.24%
  - Non-clickbait probability: 3.76%
----------------------------------------------------------------------

Text: New Study Shows Benefits of Regular Exercise
Prediction: Non-Clickbait
Confidence: 95.65%
  - Clickbait probability: 4.35%
  - Non-clickbait probability: 95.65%
----------------------------------------------------------------------

Text: 10 Shocking Celebrity Secrets They Don't Want You To Know!
Prediction: Clickbait
Confidence: 98.50%
  - Clickbait probability: 98.50%
  - Non-clickbait probability: 1.50%
----------------------------------------------------------------------

Text: Federal Reserve Announces Interest Rate Decision
Prediction: Non-Clickbait
Confidence: 99.42%
  - Clickbait probability: 0.58%
  - Non-clickbait probability: 99.42%
----------------------------------------------------------------------
